# 06 · Static scanner — horizon lists from the saved model

**In:** the checkpoint written by notebook 05 (`horizon_scan_static.pt`).
**Out:** the files the viewer reads, for any SInAS location: ranked lists,
feature contributions, GNN saliency maps and the model card.

---

## What this notebook does

Notebook 05 trains the model, which takes hours. Everything that is needed
to *use* the trained model is saved in one file, the **static checkpoint**:
the weights of the ensemble, the species embeddings, every location and
trade input, the known ranges and the settings. This notebook loads that
file and makes lists. **No training and none of the data files of
notebooks 01–04 are needed**, so it runs in minutes, also on a laptop
without a GPU.

Use it to

* make lists for other locations (or for all of them),
* change the list length or how many species get an explanation,
* re-apply the **native-range and GRIIS checks** with other settings or a
  newer GRIIS file,
* refresh the **EICAT categories** in the lists after notebook 07 has
  scored missing species.

The scores are the same as those of notebook 05: both notebooks import
the model and the scanner from `horizon_core.py`.

## What a list contains

For a location, every **candidate** is scored: a species of the species
pool that has no record in the location (no SInAS record of any status
and no GBIF presence). The list is the top `EXPORT_TOP_N` candidates.

> **Read the score as a rank, not as a chance.** `probability` orders the
> species of one location. It is not a calibrated probability of
> establishment and it cannot be compared between locations.

### Checks before a list is cut

The checks run on the full ranking, **before** it is cut to
`EXPORT_TOP_N`, so a dropped species is replaced by the next candidate.

| Check | What it finds | Default action |
|---|---|---|
| Native in the location (`CHECK_NATIVE`) | SInAS native record in the location itself (a consistency check: it should find nothing), or a record in the optional extra native checklist | drop |
| Native in the same country (`CHECK_NATIVE_SAME_COUNTRY`) | native in another SInAS location with the same ISO3 code, e.g. the mainland when an island group is scanned | flag |
| GRIIS checklist (`CHECK_GRIIS`) | already listed as introduced in the location's country; `griis_invasive` marks evidence of impact there | drop |

Actions: `"drop"` removes the candidate (and lists it in
`checks_dropped_<loc>.csv` with the reason), `"flag"` keeps it and writes
the reason to the `check_flags` column, which the viewer shows, `"off"`
skips the check.

**GRIIS works per country.** For a location that shares its ISO3 code
with other SInAS locations (a sub-national unit such as Hawaii, or its
mainland) a GRIIS record cannot be assigned to one of them. There a
GRIIS-listed species is **flagged, never dropped**
(`griis_scope = "country"`).

**Why check GRIIS when SInAS already contains it?** SInAS is a snapshot.
A newer GRIIS release, or a name that the two were matched differently
on, shows up here as a GRIIS-listed species that the model still treats
as a candidate. Look at `checks_dropped_<loc>.csv` once: each row is
either a real new introduction (correctly removed from a *horizon* list)
or a hint at a name-matching problem.

## Pipeline at a glance

| Step | What happens | Main output |
|---|---|---|
| 0 | Sources and configuration | settings |
| 1 | Load the checkpoint, show what was trained | `scanner` |
| 2 | Refresh the EICAT categories and their source | `scanner.info` |
| 3 | Build the native-range and GRIIS checks | `checks` |
| 4 | Scan, explain and write the viewer files | files in `EXPORT_DIR` |
| 5 | Look at the result: top of the list, dropped species, one species in detail | tables, plot |
| 6 | Quick ranking of any location without writing files | table |

## Outputs (`EXPORT_DIR`, default `../data/model_outputs/exports/`)

| File | One row per | Viewer use |
|---|---|---|
| `scan_<loc>.csv` | candidate species (top `EXPORT_TOP_N`) | list, filters, charts |
| `contributions_<loc>.csv` | species × feature (top `EXPLAIN_TOP_N`) | detail panel bars |
| `gnn_saliency_nodes_<loc>.csv`, `gnn_saliency_edges_<loc>.csv` | species × graph node / edge (top `SALIENCY_TOP_N`) | local saliency map |
| `checks_dropped_<loc>.csv` | candidate removed by a check | (for you to inspect) |
| `checks_summary.csv` | location | what the checks did |
| `feature_dictionary.csv` | contribution | block and label |
| `model_card.json` | model | "About the model" tab |
| `*_all_locations.csv` | — | the tables above for all scanned locations (`WRITE_COMBINED`) |

`<loc>` is the location name in lower case with underscores. Files of the
same name are **overwritten**; set `EXPORT_DIR` to another folder to keep
the lists of notebook 05. Column definitions:
`model_training_metadata.md`; sources of the checks:
`static_scanner_metadata.md`.

## How to run

1. Install (once): `pip install torch torch_geometric pandas numpy tqdm
   matplotlib` (`country_converter` only if your GRIIS file has no ISO3
   column; `pygbif` only for `GBIF_VERIFY = True`).
2. Keep `horizon_core.py` in the same folder as this notebook.
3. Make sure `../data/model_outputs/horizon_scan_static.pt` exists (run
   notebook 05 once, or copy the file from the computer that trained it).
4. Optional: download the GRIIS compendium (Step 0).
5. Set `EXPORT_LOCATIONS` in the configuration cell and *Run All*.
   One location with the default settings takes a few minutes on a CPU;
   most of the time goes into the explanations (`EXPLAIN_TOP_N`,
   `SALIENCY_TOP_N`). For all locations set these lower, or to 0 for
   lists only.

## Step 0 · Data sources and configuration

| Input | Source | How to obtain |
|---|---|---|
| `model_outputs/horizon_scan_static.pt` | Notebook 05, Step 14 | run notebook 05 (or copy the file). It is a Python pickle: **only load a checkpoint you made yourself.** |
| `horizon_core.py` | this project | keep it next to the notebook |
| `griis_country_compendium.csv` *(optional)* | Country Compendium of the Global Register of Introduced and Invasive Species — Pagad S, Bisset S, Genovesi P, Groom Q, Hirsch T, Jetz W, Ranipeta A, Schigel D, Sica YV, McGeoch MA (2022) *Scientific Data* 9: 391, [doi:10.1038/s41597-022-01514-z](https://doi.org/10.1038/s41597-022-01514-z) | **Manual download** from Zenodo, [doi:10.5281/zenodo.6348164](https://doi.org/10.5281/zenodo.6348164): take the compendium CSV of the latest version and save it under this name in `../data/`. Note the version and download date in `static_scanner_metadata.md`. Single-country checklists (also newer ones) are at <https://griis.org>. |
| `eicat_lookup.csv` *(optional)* | expert EICAT assessments (manual) and LLM-derived categories (notebook 07) | see notebook 07 |
| `native_checklist_extra.csv` *(optional)* | your own native checklist: columns `gbif_id` and `location` (SInAS name) or `iso3` | manual |
| `name_matching_audit.csv` *(optional)* | Notebook 03 | lets GRIIS names that are Backbone synonyms find their species |
| `sinas_matched_species.csv`, `complete_country_features_master.csv` *(only for older checkpoints)* | Notebooks 03 and 01 | a checkpoint of `format_version` 4 does not hold the native records and ISO3 codes; they are then read from these files |

The GRIIS file is read with whatever column names it has: the species
name (`acceptedNameUsage`, `species` or `scientificName`), the country
(`countryCode_alpha3`, or an ISO2 code or country name that is
converted) and `isInvasive`. Step 3 prints the columns it used. If it
picks the wrong ones, set `GRIIS_COLUMNS`.

In [1]:
import difflib
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from horizon_core import (ChecklistChecks, attach_eicat, export_scans,
                          load_griis, load_static_scanner,
                          plot_saliency_map, write_model_card)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

c:\Users\simon\Documents\GitHub\horizon-scanner\.venv\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


### Configuration

Normally you only change `EXPORT_LOCATIONS` and the list sizes.

In [2]:
# ---------------------------------------------------------------- paths
# Set the environment variable IAS_DATA_DIR to use another data folder.
DATA = Path(os.environ.get("IAS_DATA_DIR", Path.cwd().parent / "data"))
OUT_DIR = DATA / "model_outputs"
STATIC_CKPT = OUT_DIR / "horizon_scan_static.pt"
EXPORT_DIR = OUT_DIR / "exports"  # files of the same name are overwritten

# ---------------------------------------------------------------- lists
EXPORT_LOCATIONS = ["Belgium"]  # SInAS location names, or "all"
EXPORT_TOP_N = 500  # species per location in scan_<location>.csv
EXPLAIN_TOP_N = 100  # of these, how many get contributions (0 = none)
SALIENCY_TOP_N = 25  # of these, how many get a GNN saliency map (0 = none)
WRITE_COMBINED = True  # also write *_all_locations.csv (large for "all")

# None = the value notebook 05 exported with (stored in the checkpoint).
N_MC_SAMPLES = None  # MC-dropout passes per ensemble member
N_IG_STEPS = None  # Integrated-Gradients steps
SALIENCY_TOP_K = None  # graph nodes kept per saliency map
FEATURE_CHUNK = 4096  # pairs per chunk; lower it if memory runs out
DEVICE = None  # None = GPU if there is one, else CPU; or "cpu"

# ---------------------------------------------------------------- checks
# "drop": remove the candidate (listed in checks_dropped_<loc>.csv)
# "flag": keep it, with the reason in the check_flags column
# "off" : do not check
CHECK_NATIVE = "drop"  # native in the location (SInAS / extra checklist)
CHECK_NATIVE_SAME_COUNTRY = "flag"  # native elsewhere under the same ISO3
CHECK_GRIIS = "drop"  # on the GRIIS checklist of the location's country
GRIIS_CSV = DATA / "griis_country_compendium.csv"  # optional (Step 0)
GRIIS_COLUMNS = None  # e.g. {"iso3": "ISO3", "name": "species"}
NATIVE_EXTRA_CSV = DATA / "native_checklist_extra.csv"  # optional
NAME_AUDIT_CSV = DATA / "name_matching_audit.csv"  # optional (synonyms)
EICAT_CSV = DATA / "eicat_lookup.csv"  # optional
# Only read for checkpoints of format_version 4:
SINAS_SPECIES_CSV = DATA / "sinas_matched_species.csv"
MASTER_CSV = DATA / "complete_country_features_master.csv"

# Live GBIF check: drop candidates with any GBIF occurrence in the country
# right now (one request per candidate; slow; needs pygbif and internet).
GBIF_VERIFY = False
GBIF_COUNTRY_CODES = {"Belgium": "BE"}  # ISO-2 code per location to verify

# ---------------------------------------------------------------- look
INSPECT_RANK = 1  # Step 5: which species of the last list to show

for label, path, needed in (
        ("checkpoint", STATIC_CKPT, True), ("GRIIS", GRIIS_CSV, False),
        ("EICAT", EICAT_CSV, False),
        ("extra native checklist", NATIVE_EXTRA_CSV, False),
        ("name audit", NAME_AUDIT_CSV, False)):
    status = "OK " if path.exists() else ("MISSING" if needed else "none")
    print(f"[{status}] {label}: {path}")
if not STATIC_CKPT.exists():
    raise FileNotFoundError(
        f"{STATIC_CKPT} not found. Run notebook 05 first, or copy the "
        "checkpoint into the model_outputs folder.")

[OK ] checkpoint: c:\Users\simon\Documents\GitHub\horizon-scanner\data\model_outputs\horizon_scan_static.pt
[none] GRIIS: c:\Users\simon\Documents\GitHub\horizon-scanner\data\griis_country_compendium.csv
[none] EICAT: c:\Users\simon\Documents\GitHub\horizon-scanner\data\eicat_lookup.csv
[none] extra native checklist: c:\Users\simon\Documents\GitHub\horizon-scanner\data\native_checklist_extra.csv
[OK ] name audit: c:\Users\simon\Documents\GitHub\horizon-scanner\data\name_matching_audit.csv


### Helper functions

Two readers that are only needed for older checkpoints
(`format_version` 4), and a check of the location names.

In [3]:
def as_bool(series):
    """Convert a column of True/False, 1/0 or yes/no values to bool."""
    if series.dtype == bool:
        return series
    return (series.astype(str).str.strip().str.lower()
            .isin({"true", "1", "yes", "t", "1.0"}))


def native_pairs_from_sinas(path, info, location_names):
    """SInAS native records as (species rows, location rows).

    Same rule as notebook 05: a row counts as native when ``is_native``
    is set and ``is_alien`` is not.
    """
    sinas = pd.read_csv(path, low_memory=False,
                        usecols=["gbif_id", "location", "is_native",
                                 "is_alien"])
    native = sinas[as_bool(sinas["is_native"]) & ~as_bool(sinas["is_alien"])]
    sp_row = {int(g): i for i, g in enumerate(info["gbif_id"])}
    co_row = {name: i for i, name in enumerate(location_names)}
    sp = native["gbif_id"].map(sp_row)
    co = native["location"].astype(str).str.strip().map(co_row)
    ok = sp.notna() & co.notna()
    pairs = np.unique(np.stack([sp[ok].to_numpy(dtype=np.int64),
                                co[ok].to_numpy(dtype=np.int64)]), axis=1)
    return pairs[0], pairs[1]


def iso3_from_master(path, location_names):
    """ISO3 code per location from the master table of notebook 01."""
    master = pd.read_csv(path, low_memory=False,
                         usecols=["location", "ISO3"])
    master["location"] = master["location"].astype(str).str.strip()
    codes = (master.drop_duplicates("location").set_index("location")["ISO3"]
             .reindex(location_names))
    return [code.strip().upper() if isinstance(code, str)
            and len(code.strip()) == 3 else None for code in codes]


def resolve_locations(wanted, known):
    """Check location names; suggest close matches for unknown ones."""
    if wanted == "all":
        return list(known)
    unknown = [name for name in wanted if name not in known]
    if unknown:
        hints = {name: difflib.get_close_matches(name, known, n=3)
                 for name in unknown}
        raise ValueError(f"Unknown SInAS location name(s): {hints} "
                         "(name: closest known names)")
    return list(wanted)

## Step 1 · Load the checkpoint

`load_static_scanner` rebuilds the ensemble and its feature builder from
the checkpoint and returns a `Scanner`. The cell then prints what the
checkpoint holds, so you can confirm it is the model you mean: when it
was trained, with which main settings, and how it did in the
cross-validation of notebook 05.

Two warnings are raised when they apply:

* the model was trained with `LIMITED_DATA_CONTROLS = False` (an
  ablation run without the effort correction; do not use its lists);
* the checkpoint was a quick test run (`QUICK_RUN = True`).

In [4]:
print("Step 1: loading the checkpoint")
scanner, ckpt = load_static_scanner(
    STATIC_CKPT, device=DEVICE, feature_chunk=FEATURE_CHUNK,
    n_mc_samples=N_MC_SAMPLES, n_ig_steps=N_IG_STEPS,
    saliency_top_k=SALIENCY_TOP_K)
meta, config = ckpt["meta"], ckpt["config"]
location_names = list(meta["all_country_names"])
print(f"   trained: {meta.get('trained_at')} | checkpoint format "
      f"{ckpt['format_version']} | device: {scanner.device}")
print(f"   ensemble: {len(scanner.members)} model(s) | species in graph: "
      f"{len(scanner.info):,} | candidate species: "
      f"{int(scanner.eligible.sum()):,} | locations: {len(location_names)}")
print(f"   effort correction: {meta.get('effort_correction')} | climate "
      f"gate: {meta.get('climate_gate')} | species pool: "
      f"{config.get('SPECIES_POOL')}")
print(f"   per list: {scanner.n_mc_samples} MC-dropout passes per model, "
      f"{scanner.n_ig_steps} IG steps, {scanner.saliency_top_k} nodes per "
      "saliency map")
if config.get("LIMITED_DATA_CONTROLS") is False:
    warnings.warn("This model was trained with LIMITED_DATA_CONTROLS = "
                  "False (ablation run, no effort correction). Retrain "
                  "with True before using the lists.")
if config.get("QUICK_RUN"):
    warnings.warn("This checkpoint comes from a QUICK_RUN of notebook 05; "
                  "its scores are not meaningful.")

# --- cross-validation results stored with the model ------------------------
rows = []
for split, label in (("random_pairs", "random pairs"),
                     ("species_held_out", "species held out"),
                     ("temporal", "temporal hold-out")):
    result = ckpt["metrics"].get(split)
    if result:
        rows.append({"test": label, **{
            key: result[key]["mean"] for key in
            ("roc_auc", "pr_auc", "rank_hit_top10pct",
             "rank_median_top_frac") if isinstance(result.get(key), dict)}})
if rows:
    print("\n   cross-validation (means over folds; see notebook 05):")
    print(pd.DataFrame(rows).round(3).to_string(index=False))
else:
    print("\n   no cross-validation results in this checkpoint")

locations = resolve_locations(EXPORT_LOCATIONS, location_names)
print(f"\n   locations to scan: {len(locations)}")

Step 1: loading the checkpoint
   trained: 2026-11-10T08:03:58+00:00 | checkpoint format 4 | device: cpu
   ensemble: 3 model(s) | species in graph: 173,249 | candidate species: 32,670 | locations: 289
   effort correction: none | climate gate: none | species pool: sinas_alien
   per list: 30 MC-dropout passes per model, 50 IG steps, 15 nodes per saliency map

   no cross-validation results in this checkpoint

   locations to scan: 1


C:\Users\simon\AppData\Local\Temp\ipykernel_28864\2138118855.py:20: UserWarning: This model was trained with LIMITED_DATA_CONTROLS = False (ablation run, no effort correction). Retrain with True before using the lists.
  warnings.warn("This model was trained with LIMITED_DATA_CONTROLS = "


## Step 2 · Refresh the EICAT categories

The checkpoint holds the EICAT categories that were known when the model
was trained. `eicat_lookup.csv` may have grown since (new expert
assessments, or LLM-derived categories from notebook 07), so it is read
again here.

Each category carries its **source**:

| `eicat_source` | Meaning |
|---|---|
| `expert` | a published EICAT assessment (IUCN / GISD, entered by hand) |
| `llm` | derived by a language model in notebook 07; `eicat_model`, `eicat_confidence` and `eicat_justification` say by which model and on what grounds. **Not an assessment**: a pointer for where to look first. |
| `none` | no category (`eicat = "NE"`, not evaluated) |

The model does not use EICAT; it is shown next to the risk only.

In [5]:
print("Step 2: EICAT categories")
scanner.info = attach_eicat(scanner.info, EICAT_CSV)
pool_info = scanner.info[scanner.eligible.cpu().numpy()]
print("   candidate species by EICAT source:")
print(pd.crosstab(pool_info["eicat_source"], pool_info["eicat"])
      .to_string())
if not EICAT_CSV.exists():
    print(f"   ({EICAT_CSV.name} not found: every species is NE)")

Step 2: EICAT categories
   candidate species by EICAT source:


C:\Users\simon\AppData\Local\Temp\ipykernel_28864\1444949922.py:2: FutureWarning: Impact data is no longer written into the lists; the viewer reads species_impacts_master.csv from notebook 07. attach_eicat() can be removed.
  scanner.info = attach_eicat(scanner.info, EICAT_CSV)


KeyError: 'eicat_source'

## Step 3 · Build the native-range and GRIIS checks

**What:**

1. The SInAS native records and the ISO3 code of every location come
   from the checkpoint. A checkpoint of `format_version` 4 does not hold
   them; they are then read from the files of notebooks 03 and 01, and
   the check is skipped (with a message) if those files are missing.
2. The GRIIS compendium is read and its names are matched to the
   species of the model: by canonical name (genus + species epithet,
   authorship dropped), against the accepted Backbone names and, if
   `name_matching_audit.csv` is there, against every synonym that
   notebook 03 resolved. The cell prints how many GRIIS rows were
   matched. Rows that are not matched are mostly species outside the
   model's species list (GRIIS species that SInAS lacks), which cannot
   be candidates anyway.
3. `ChecklistChecks` is set up with the three actions.

**Check the output:** a low matched share (well below ~70 %) or a wrong
column under "columns used" means the file was not read as intended.

In [ ]:
print("Step 3: native-range and GRIIS checks")
native_pairs, location_iso3 = ckpt["native_pairs"], ckpt["location_iso3"]
if native_pairs is None and SINAS_SPECIES_CSV.exists():
    native_pairs = native_pairs_from_sinas(SINAS_SPECIES_CSV, scanner.info,
                                           location_names)
    print(f"   native records read from {SINAS_SPECIES_CSV.name}")
if location_iso3 is None and MASTER_CSV.exists():
    location_iso3 = iso3_from_master(MASTER_CSV, location_names)
    print(f"   ISO3 codes read from {MASTER_CSV.name}")
if native_pairs is None:
    print("   no native records available: the native checks are skipped")
if location_iso3 is None:
    print("   no ISO3 codes available: the GRIIS and same-country checks "
          "are skipped")

griis = None
if CHECK_GRIIS != "off" and GRIIS_CSV.exists() and location_iso3:
    griis = load_griis(GRIIS_CSV, scanner.info, audit_csv=NAME_AUDIT_CSV,
                       columns=GRIIS_COLUMNS)
    report = griis.attrs["report"]
    print(f"   GRIIS: {report['rows']:,} rows | matched to a species of "
          f"the model: {report['rows_matched_to_a_species']:,} "
          f"({report['share_matched']:.0%}) | {report['countries']} "
          f"countries | {report['species']:,} species")
    print(f"   columns used: {report['columns_used']}")
elif CHECK_GRIIS != "off":
    print(f"   GRIIS check skipped ({GRIIS_CSV.name} not found, or no ISO3 "
          "codes); see Step 0 for the download")

extra_native = (pd.read_csv(NATIVE_EXTRA_CSV) if NATIVE_EXTRA_CSV.exists()
                else None)
checks = ChecklistChecks(
    scanner.info, location_names, location_iso3, native_pairs, griis,
    extra_native, native_action=CHECK_NATIVE,
    same_country_action=CHECK_NATIVE_SAME_COUNTRY, griis_action=CHECK_GRIIS)
print(f"   native records: {checks.report['native_records']:,} | extra "
      f"native records: {checks.report['extra_native_records']:,} | "
      f"actions: {checks.report['actions']}")
no_code = checks.report["locations_without_iso3"]
if no_code and location_iso3:
    print(f"   {len(no_code)} location(s) without an ISO3 code (no GRIIS "
          f"check there): {no_code[:8]}")

## Step 4 · Scan, explain and write the viewer files

For every location in `EXPORT_LOCATIONS`:

1. **Rank** all candidates with the ensemble (scan mode: documentation
   effort set to that of a typical known invader).
2. **Apply the checks** to the full ranking, then keep the top
   `EXPORT_TOP_N`.
3. **Uncertainty** for the list: MC dropout in every ensemble member.
4. **Contributions** for the top `EXPLAIN_TOP_N` (Integrated Gradients)
   and **GNN saliency maps** for the top `SALIENCY_TOP_N`.
5. **Write** the CSV files, and once the feature dictionary, the checks
   summary and the model card.

The model card is the one stored in the checkpoint, with two additions:
what the checks did (`checks`) and how this export was made (`scan`).

In [ ]:
print("Step 4: scanning")
exported = export_scans(
    scanner, locations, EXPORT_DIR, top_n=EXPORT_TOP_N,
    explain_n=EXPLAIN_TOP_N, saliency_n=SALIENCY_TOP_N, checks=checks,
    gbif_codes=GBIF_COUNTRY_CODES if GBIF_VERIFY else None,
    write_combined=WRITE_COMBINED)

model_card = ckpt["model_card"] or {
    # older checkpoints carry no model card: the essentials
    "trained_at": meta.get("trained_at"),
    "n_species_in_graph": len(scanner.info),
    "n_candidate_species": int(scanner.eligible.sum()),
    "n_locations": len(location_names),
    "ensemble_size": len(scanner.members),
    "metrics": ckpt["metrics"],
}
card_path = write_model_card(
    model_card, EXPORT_DIR, checks=checks.report,
    scan={"source": "06_static_scanner.ipynb",
          "checkpoint": STATIC_CKPT.name, "locations": len(locations),
          "top_n": EXPORT_TOP_N, "explain_n": EXPLAIN_TOP_N,
          "saliency_n": SALIENCY_TOP_N, "gbif_verify": GBIF_VERIFY,
          "eicat_file": EICAT_CSV.name if EICAT_CSV.exists() else None})
print(f"   wrote the lists of {len(locations)} location(s) and "
      f"{card_path.name} to {EXPORT_DIR}")

## Step 5 · Look at the result

**5a · What the checks did.** One row per location. `dropped_from_top`
is the number of candidates removed from what would have been the list;
`flagged_in_list` the number of listed species that carry a flag.

In [ ]:
summary = exported["summary"]
if len(summary):
    print(summary.to_string(index=False))
else:
    print("no checks were applied")

**5b · Top of the last list**, and the species the checks removed from
it. In the viewer the same list can be filtered and sorted.

In [ ]:
last = exported["last"]
scan = last["scan"]
show = ["rank", "taxon_name", "group", "probability", "uncertainty",
        "data_support_class", "effort_shift", "eicat", "eicat_source",
        "check_flags", "top_driver"]
print(f"Top 15 for {scan['target_country'].iat[0]}:")
print(scan[[c for c in show if c in scan.columns]].head(15)
      .round(3).to_string(index=False))
dropped = last["checks_dropped"]
if len(dropped):
    print(f"\nRemoved by the checks ({len(dropped)}):")
    print(dropped.drop(columns="target_country").head(20).round(3)
          .to_string(index=False))

**5c · One species in detail** (`INSPECT_RANK`): its largest
contributions and, if it has one, its GNN saliency map. Red raises the
risk, blue lowers it; contributions are in logit units relative to an
average candidate.

In [ ]:
row = scan.iloc[min(INSPECT_RANK, len(scan)) - 1]
gbif_id = int(row["gbif_id"])
print(f"{row['taxon_name']} -> {row['target_country']}: rank "
      f"{int(row['rank'])}, score {row['probability']:.3f} "
      f"+/- {row['uncertainty']:.3f}, data support "
      f"{row['data_support_class']}, EICAT {row['eicat']} "
      f"({row['eicat_source']})")
contributions = last["contributions"]
if len(contributions):
    own = contributions[contributions["gbif_id"] == gbif_id]
    if len(own):
        top = own.loc[own["value"].abs().nlargest(12).index,
                      ["block", "name", "value"]]
        print(top.round(3).to_string(index=False))
    else:
        print("no contributions for this species (raise EXPLAIN_TOP_N)")
if len(last["saliency_nodes"]):
    plot_saliency_map(last["saliency_nodes"], last["saliency_edges"],
                      gbif_id)

## Step 6 · Quick ranking of any location (optional)

`quick_list` ranks the candidates of one location and applies the checks,
without uncertainty, explanations or files. It takes seconds and is
handy for comparing locations or check settings before a full export.

In [ ]:
def quick_list(location, n=20):
    """Top ``n`` candidates of a location after the checks (no files)."""
    ranked = scanner.rank(location)
    kept, _ = checks.apply(ranked, location, scanner.info, n)
    checks.log.pop()  # keep the summary of Step 4 as it is
    info = scanner.info.loc[kept["sp_idx"].head(n),
                            ["taxon_name", "group", "data_support_class",
                             "eicat", "eicat_source"]]
    out = pd.concat([info.reset_index(drop=True),
                     kept[["score_deterministic", "check_flags"]].head(n)
                     .reset_index(drop=True)], axis=1)
    out.insert(0, "rank", np.arange(1, len(out) + 1))
    return out


quick_list(locations[0], 20)